# LangChain Evaluator Tracer Reference

Developer-facing statements defined in `langchain_core.tracers.evaluation`.

# `wait_for_all_evaluators`

Waits for all evaluator callback handlers that are still alive to finish their pending evaluator tasks.

```python
wait_for_all_evaluators(
) -> None
```

The module tracks handlers through weak references, so handlers that have already been garbage-collected are not retained.

---

# `EvaluatorCallbackHandler: BaseTracer`

Synchronous tracer that runs one or more LangSmith run evaluators whenever a completed root run is persisted.

## Fields

```python
name: str = "evaluator_callback_handler" # Callback-handler name
example_id: UUID | None = None # Example ID associated with evaluated runs
client: langsmith.Client # LangSmith client used for evaluation and feedback
evaluators: Sequence[langsmith.RunEvaluator] = () # Evaluators applied to persisted runs
executor: ThreadPoolExecutor | None = None # Executor used to run evaluators concurrently
futures: weakref.WeakSet[Future[None]] = weakref.WeakSet() # Pending evaluator futures
skip_unfinished: bool = True # Whether runs without outputs are skipped
project_name: str | None = None # Project used for evaluator-generated traces
logged_eval_results: dict[tuple[str, str], list[EvaluationResult]] # Evaluation results grouped by target run and example
lock: threading.Lock # Lock protecting logged evaluation results
```

## Constructor

```python
EvaluatorCallbackHandler(
    evaluators: Sequence[langsmith.RunEvaluator], # Evaluators applied to all top-level runs
    client: langsmith.Client | None = None, # LangSmith client; created through the tracer client helper when omitted
    example_id: UUID | str | None = None, # Example ID associated with evaluated runs
    skip_unfinished: bool = True, # Whether runs without outputs are skipped
    project_name: str | None = "evaluators", # Project used for evaluator-generated traces
    max_concurrency: int | None = None, # Maximum number of concurrent evaluator tasks
    **kwargs: Any, # Arguments forwarded to BaseTracer
) -> None
```

String `example_id` values are converted to `UUID`.

When `max_concurrency` is `None`, the handler uses LangChain's shared tracer executor. When it is greater than zero, the handler creates a dedicated `ThreadPoolExecutor` with that worker limit and registers a finalizer that shuts it down with `wait=True`. When it is zero or negative, evaluation runs synchronously without an executor.

## Methods

### `wait_for_futures`

Waits for every evaluator future currently tracked by the handler.

```python
wait_for_futures(
    self,
) -> None
```

## Behaviour

As a `BaseTracer` subclass, the handler evaluates completed root runs when they are persisted. Child runs are not independently persisted by the base tracer.

When `skip_unfinished` is `True`, a run with no outputs is skipped. Otherwise, the handler copies the run, assigns `example_id` to the copy's `reference_example_id`, and applies every configured evaluator.

When `project_name` is `None`, evaluation delegates to `client.evaluate_run()`. When a project name is configured, evaluation runs inside `tracing_v2_enabled()` with the `"eval"` tag. A referenced example is loaded through the client when the run has a reference example ID, and resulting feedback is written through `client.create_feedback()`.

Completed evaluation results are stored in `logged_eval_results` under `(target_run_id, reference_example_id)` string keys. Access to this mapping is protected by `lock`.

Evaluator exceptions are logged and re-raised. An evaluator response that is neither an `EvaluationResult` nor an `EvaluationResults` dictionary containing `"results"` raises `TypeError`.

In [ ]:
from typing import Any # Import Any for flexible method parameters

from langchain_core.runnables import RunnableLambda # Import a runnable for creating a traced operation
from langchain_core.tracers.evaluation import EvaluatorCallbackHandler, wait_for_all_evaluators # Import evaluator tools
from langsmith.evaluation.evaluator import EvaluationResult # Import the evaluation result model


class OutputEvaluator: # Create a simple evaluator
    def evaluate_run( # Evaluate one completed LangChain run
        self,
        run: Any, # Receive the completed run
        example: Any = None, # Accept an optional reference example
    ) -> EvaluationResult:
        has_output = bool(run.outputs) # Check whether the run produced output

        return EvaluationResult( # Return an evaluation result
            key="has_output", # Name the evaluation
            score=1 if has_output else 0, # Give one point when output exists
            comment="The run produced output." if has_output else "The run produced no output.", # Explain the score
        ) # Finish creating the result


class FakeLangSmithClient: # Create a client that sends no network requests
    def evaluate_run( # Evaluate a run using the supplied evaluator
        self,
        run: Any, # Receive the completed run
        evaluator: OutputEvaluator, # Receive the evaluator
    ) -> EvaluationResult:
        return evaluator.evaluate_run(run) # Run the evaluator locally


handler = EvaluatorCallbackHandler( # Create the evaluator callback
    evaluators=[OutputEvaluator()], # Register the evaluator
    client=FakeLangSmithClient(), # Use the local fake client
    project_name=None, # Avoid creating evaluator traces
    max_concurrency=0, # Run evaluation synchronously
) # Finish creating the handler

square = RunnableLambda(lambda number: number * number) # Create a traced operation

result = square.invoke( # Run the operation
    5, # Provide the input
    config={ # Configure tracing
        "callbacks": [handler], # Attach the evaluator callback
        "run_name": "square_operation", # Set the run name
    },
) # Finish invoking the runnable

handler.wait_for_futures() # Wait for this handler's evaluator tasks
wait_for_all_evaluators() # Wait for all active evaluator handlers

print("Runnable result:", result) # Display the runnable result
print("Logged evaluation groups:", len(handler.logged_eval_results)) # Display evaluated run count

for key, evaluations in handler.logged_eval_results.items(): # Visit every evaluated run
    run_id, example_id = key # Separate the run and example IDs

    print("\nTarget run ID:", run_id) # Display the evaluated run ID
    print("Reference example ID:", example_id) # Display the reference example ID

    for evaluation in evaluations: # Visit every evaluation result
        print("Evaluation key:", evaluation.key) # Display the evaluation name
        print("Score:", evaluation.score) # Display the score
        print("Comment:", evaluation.comment) # Display the explanation